In [ ]:
import numpy as np
print(np.__version__)


In [ ]:
import pandas as pd
print(pd.__version__)


In [ ]:
import numpy as np
import pandas as pd

Loading the dataset 

In [ ]:
heart_df=pd.read_csv(r'C:\Users\Malapreethi\Downloads\heart (1).csv')
heart_df

In [ ]:
heart_df.sample(5)

In [ ]:
heart_df.info()

In [ ]:
heart_df.describe()

In [ ]:
heart_df.describe(include="all")

Preprocessing of data in the acquired dataset

In [ ]:
#data preprocessing
heart_df.isnull()

In [ ]:
heart_df.duplicated().sum()

In [ ]:
heart_df.isnull().sum()

In [ ]:
heart_df.nunique()

In [ ]:
heart_df.columns

Converting the values present in the categorical variables into numerical values

In [ ]:
cat_col = heart_df.select_dtypes(include='object').columns
cat_col

In [ ]:
#converting categorical variables to numeric
for col in cat_col:
    print(col)
    print((heart_df[col].unique()),list(range(heart_df[col].nunique())))
    heart_df[col].replace((heart_df[col].unique()),range(heart_df[col].nunique()),inplace=True)
    print('*'*90)
    print()

Dataset after converting the values in the categorical variables into numerical values

In [ ]:
heart_df

Filtering out the null values present in the 'RestingBP' and 'Cholesterol' columns respectively using the KNN algorithm

In [ ]:
heart_df['Cholesterol'].value_counts()

Replacing the null values present in cholesterol column with 'NaN'

In [ ]:
heart_df['Cholesterol'].replace(0,np.nan,inplace=True) 

Replacing the NaN values present in the cholesterol column

In [ ]:
from sklearn.impute import KNNImputer

In [ ]:
imputer=KNNImputer(n_neighbors=3) #creating an object of KNNImputer
after_impute=imputer.fit_transform(heart_df)
heart_df=pd.DataFrame(after_impute,columns=heart_df.columns) #NaN values present inthe dataset is replaced with calculted KNN value


Checking if every NaN value in the cholesterol column is replaced

In [ ]:
heart_df['Cholesterol'].isna().sum()

Checking if any null value is present in RestingBP column

In [ ]:
heart_df['RestingBP'][heart_df['RestingBP']==0]

In [ ]:
from sklearn.impute import KNNImputer
heart_df['RestingBP'].replace(0,np.nan,inplace=True)
imputer=KNNImputer(n_neighbors=3)
after_impute=imputer.fit_transform(heart_df)
heart_df=pd.DataFrame(after_impute,columns=heart_df.columns)

In [ ]:
heart_df['RestingBP'].isnull().sum()

Changing the data types of every columns into int

In [ ]:
withoutOldPeak=heart_df.columns
withoutOldPeak=withoutOldPeak.drop('Oldpeak') #Oldpeak column has 'floating' datatype
heart_df[withoutOldPeak]=heart_df[withoutOldPeak].astype('int32')

In [ ]:
heart_df.info()

DATA VISUALISATION


In [ ]:
import plotly.express as px

Finding out the correlation between the features

In [ ]:
heart_df.corr()

Finding out the correlation of every features with 'HeartDisease' feature

In [ ]:
heart_df.corr()['HeartDisease'][:-1].sort_values()

Plotting the correlation between every feature and 'HeartDisease' feature using Plotly library

In [ ]:
px.line(heart_df.corr()['HeartDisease'][:-1].sort_values())

Visualising the distribution between Heartdisease feature and Age feature using Sunburst & Histogram

In [ ]:
px.sunburst(heart_df,path=['HeartDisease','Age'])

In [ ]:
px.histogram(heart_df,x='Age',color='HeartDisease')

Visualising heart disease prediction using pie chart

In [ ]:
px.pie(heart_df,names='HeartDisease',title='Heart disease distribution')

Distribution between sex feature and heart disease feature

In [ ]:
px.histogram(heart_df,x='Sex',color='HeartDisease')

Relationship between Chest pain type & Heart Disease

In [ ]:
px.histogram(heart_df,x='ChestPainType',color='HeartDisease')


Relationship between RestingBP and HeartDisease

In [ ]:
px.sunburst(heart_df,path=['HeartDisease','RestingBP'])

Relationship between FastingBS and Heart disease

In [ ]:
px.histogram(heart_df,x='FastingBS',color='HeartDisease')

Relationship between MaxHR and Heart Disease

In [ ]:
px.sunburst(heart_df,path=['HeartDisease','MaxHR'])

In [ ]:
px.violin(heart_df,x='HeartDisease',y='MaxHR',color='HeartDisease')

Relationship between Oldpeak and heart disease

In [ ]:
px.violin(heart_df,x='HeartDisease',y='Oldpeak',color='HeartDisease')

Relationship between ST slope and Heart disease

In [ ]:
px.histogram(heart_df,x='ST_Slope',color='HeartDisease')

Relationship between Exercise Angina and Heart disease

In [ ]:
px.histogram(heart_df,x='ExerciseAngina',color='HeartDisease')

MODEL TRAINING

Splitting the data into training data and testing data

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    heart_df.drop('HeartDisease',axis=1),
    heart_df['HeartDisease'],   #declaring heart disease as the target column
    test_size=0.2,    #20% of the dataset will be used for testing
    random_state=42,
    stratify=heart_df['HeartDisease']
)

Logistic Regression:

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report,confusion_matrix
from yellowbrick.classifier import ConfusionMatrix
from sklearn.metrics import roc_curve,roc_auc_score,RocCurveDisplay
solver=['lbfgs','liblinear','newton-cg','newton-cholesky','sag','saga']  #default solvers of logistic regression algorithm
best_solver=''           #to store the best solver among the 6 given solvers
test_score=np.zeros(6)     #creating an array with 6 elements
for i,n in enumerate(solver):       #i is index value and n is the name of the solver, identifying the best solver
    lr=LogisticRegression(solver=n).fit(X_train,y_train)
    test_score[i]=lr.score(X_test, y_test)
    if lr.score(X_test, y_test)==test_score.max():
        best_solver=n
print("Best solver:",best_solver)
lr=LogisticRegression(solver=best_solver)           #training the model
lr.fit(X_train, y_train)
lr_pred=lr.predict(X_test)
lr_pred_prob=lr.predict_proba(X_test)[:,1]          
accuracy=accuracy_score(y_test,lr_pred)               #calculating the metrics values of the model
precision=precision_score(y_test,lr_pred)
recall=recall_score(y_test,lr_pred)
f1=f1_score(y_test,lr_pred)
auc=roc_auc_score(y_test,lr_pred_prob)
report=classification_report(y_test,lr_pred,output_dict=True)
print(f"Accuracy value:{accuracy:.4f}")
print(f"Precision value:{precision:.4f}")
print(f"Recall value:{recall:.4f}")
print(f"F1 score:{f1:.4f}")
print(f"AUC score:{auc:.4f}")
print("Visualising confusion matrix:")                     #visualising confusion matrix
visualizer=ConfusionMatrix(lr,classes=[0,1])
visualizer.fit(X_train,y_train)
visualizer.score(X_test,y_test)
visualizer.show()
print("Visualising ROC curve:")                           #visualising roc curve
RocCurveDisplay.from_estimator(lr,X_test,y_test)
plt.plot([0, 1],[0, 1],color='red',linestyle='--') 
plt.title("ROC Curve")
plt.show()
plt.figure(figsize=(8, 6))                             #visualising classification report
sns.heatmap(pd.DataFrame(report).iloc[:-1, :].T,annot=True,cmap='Blues')
plt.title("Logistic regression classification report")
print("Visualising classification report:")
plt.show()
results=[]
results.append({
        "Algorithm":"Logistic regression",
        "Accuracy":accuracy,
        "Precision":precision,
        "Recall":recall,
        "F1 score":f1
})

In [ ]:
import pickle
file=open('logisticR.pkl','wb')
pickle.dump(lr,file)  #lr is the variable in which the model is stored

Support vector machine

In [ ]:
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report,confusion_matrix
import seaborn as sns
from yellowbrick.classifier import ClassificationReport
iris=load_iris()
kernels={'linear':0,'poly':0,'rbf':0,'sigmoid':0}  #kernels are mathematical functions which manipulates the data
best=''
for i in kernels:
    svm=SVC(kernel=i)      #creating an instance of svc classifier
    svm.fit(X_train,y_train)
    yhat=svm.predict(X_test)
    kernels[i]=f1_score(y_test,yhat,average="weighted")
    if kernels[i]==max(kernels.values()):
        best=i
print("Best kernel:",best)
svm=SVC(kernel=best,probability=True)
svm.fit(X_train,y_train)
svm_pred=svm.predict(X_test)
svm_pred_prob=svm.predict_proba(X_test)[:,1]
accuracy=accuracy_score(y_test,svm_pred)
precision=precision_score(y_test,svm_pred)
recall=recall_score(y_test,svm_pred)
f1=f1_score(y_test,svm_pred)
auc=roc_auc_score(y_test,svm_pred_prob)
confusion_matrix=confusion_matrix(y_test,svm_pred)
report=classification_report(y_test,svm_pred)
print(f"Accuracy value:{accuracy:.4f}")
print(f"Precision value:{precision:.4f}")
print(f"Recall value:{recall:.4f}")
print(f"F1 score:{f1:.4f}")
print(f"AUC score:{auc:.4f}")
print("Confusion matrix:")
sns.heatmap(confusion_matrix,annot=True,fmt='d',cmap='Blues')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Support vector machine confusion matrix')
plt.show()
print("Classification report:")
visualizer=ClassificationReport(svm,classes=iris.target_names,support=True)
visualizer.fit(X_train,y_train)        
visualizer.score(X_test,y_test)        
visualizer.show()    
results.append({
        "Algorithm":"Support vector machine",
        "Accuracy":accuracy,
        "Precision":precision,
        "Recall":recall,
        "F1 score":f1
})  

In [ ]:
import pickle
file=open('svm.pkl','wb')
pickle.dump(svm,file)

Decision tree classifier

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.tree import plot_tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV     #used to carry out all possible iterations of the given parameters
from sklearn import metrics
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report,confusion_matrix
dtree=DecisionTreeClassifier(class_weight='balanced')
param_grid={'max_depth':[3,4,5,6,7,8],
            'min_samples_split':[2,3,4],
            'min_samples_leaf':[1,2,3,4],
            'random_state':[0,42]
}
grid_search=GridSearchCV(dtree,param_grid,cv=5)
grid_search.fit(X_train, y_train)
Ctree=DecisionTreeClassifier(**grid_search.best_params_,class_weight='balanced')     #to access the best set of parameters
Ctree.fit(X_train,y_train)
dtc_pred=Ctree.predict(X_test)
dtc_pred_prob=Ctree.predict_proba(X_test)[:,1]
accuracy=accuracy_score(y_test,dtc_pred)
precision=precision_score(y_test,dtc_pred)
recall=recall_score(y_test,dtc_pred)
f1=f1_score(y_test,dtc_pred)
auc=roc_auc_score(y_test,dtc_pred_prob)
confusion_matrix=confusion_matrix(y_test,dtc_pred)
report=classification_report(y_test,dtc_pred,output_dict=True)
print(f"Accuracy value:{accuracy:.4f}")
print(f"Precision value:{precision:.4f}")
print(f"Recall value:{recall:.4f}")
print(f"F1 score:{f1:.4f}")
print(f"AUC score:{auc:.4f}")
plt.figure(figsize=(12, 8))
plot_tree(Ctree)
print("Visualising the decision tree:")
plt.title("Decision tree visualisation")
plt.show()
print("Confusion matrix visualisation:")
sns.heatmap(confusion_matrix,annot=True,fmt='d',cmap='Blues',xticklabels=iris.target_names,yticklabels=iris.target_names)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title("Decision tree confusion matrix")
plt.show()
df_report = pd.DataFrame(report).transpose()
df_report = df_report.drop(columns=['support'],errors='ignore')
plt.figure(figsize=(10, 6))
sns.heatmap(df_report.iloc[:-3, :],annot=True,cmap='Blues',fmt='.2f')
print("Classification report:")
plt.title("Decision tree classification report")
plt.show()
results.append({
        "Algorithm":"Decision tree classifier",
        "Accuracy":accuracy,
        "Precision":precision,
        "Recall":recall,
        "F1 score":f1
})

In [ ]:
import pickle
file=open('decisiontree.pkl','wb')
pickle.dump(Ctree,file)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay
data=load_iris()
rfc=RandomForestClassifier()
param_grid={
    'n_estimators':[50,100,150,500],
    'max_features':['sqrt','log2',None],
    'max_depth':[3,6,9,19],
    'max_leaf_nodes':[3,6,9],
}
grid_search=GridSearchCV(rfc,param_grid)
grid_search.fit(X_train,y_train)
rfctree=RandomForestClassifier(**grid_search.best_params_)
rfctree.fit(X_train,y_train)
rfc_pred=rfctree.predict(X_test)
rfc_pred_prob=rfctree.predict_proba(X_test)[:,1]
accuracy=accuracy_score(y_test,rfc_pred)
precision=precision_score(y_test,rfc_pred)
recall=recall_score(y_test,rfc_pred)
f1=f1_score(y_test,rfc_pred)
auc=roc_auc_score(y_test,rfc_pred_prob)
confusion_matrix=confusion_matrix(y_test,rfc_pred)
report=classification_report(y_test,rfc_pred,output_dict=True)
print(f"Accuracy value:{accuracy:.4f}")
print(f"Precision value:{precision:.4f}")
print(f"Recall value:{recall:.4f}")
print(f"F1 score:{f1:.4f}")
print(f"AUC score:{auc:.4f}")
print("\nConfusion matrix:\n",confusion_matrix)
print("Confusion matrix:")
disp = ConfusionMatrixDisplay.from_predictions(y_test,rfc_pred,cmap=plt.cm.Blues)
plt.title("Random forest confusion matrix")
plt.show()
print("Classification report:")
df_report = pd.DataFrame(report).iloc[:-1, :].T 
plt.figure(figsize=(10, 6))
sns.heatmap(df_report, annot=True, cmap='Blues', fmt='.2f')
plt.title('Random Forest Classification Report')
plt.show()
results.append({
        "Algorithm":"Random forest classifier",
        "Accuracy":accuracy,
        "Precision":precision,
        "Recall":recall,
        "F1 score":f1
})

In [ ]:
import pickle
file=open('randomforest.pkl','wb')
pickle.dump(rfctree,file) 

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report,confusion_matrix
model=xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    use_label_encoder=False,
    eval_metric='logloss'
)
model.fit(X_train,y_train)
xg_pred=model.predict(X_test)
xg_pred_prob=model.predict_proba(X_test)[:,1]
accuracy=accuracy_score(y_test,xg_pred)
precision=precision_score(y_test,xg_pred)
recall=recall_score(y_test,xg_pred)
f1=f1_score(y_test,xg_pred)
auc=roc_auc_score(y_test,xg_pred_prob)
confusion_matrix=confusion_matrix(y_test,xg_pred)
report=classification_report(y_test,xg_pred,output_dict=True)
print(f"Accuracy value:{accuracy:.4f}")
print(f"Precision value:{precision:.4f}")
print(f"Recall value:{recall:.4f}")
print(f"F1 score:{f1:.4f}")
print(f"AUC score:{auc:.4f}")
print("\nConfusion matrix:\n",confusion_matrix)
print("Visualising feature importance:")                      #feature importance
xgb.plot_importance(model)
plt.show()
print("Visualising confusion matrix:")
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix,annot=True,fmt='g',cmap='Blues', 
            xticklabels=load_iris().target_names, 
            yticklabels=load_iris().target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion matrix heatmap')
plt.show()
print("Visualising classification report:")
report = classification_report(y_test, xg_pred, output_dict=True)
sns.heatmap(pd.DataFrame(report).iloc[:-1, :].T, annot=True, cmap='Blues')
plt.title("Classification Report")
plt.show()
results.append({
        "Algorithm":"XGBoost",
        "Accuracy":accuracy,
        "Precision":precision,
        "Recall":recall,
        "F1 score":f1
})

In [ ]:
import pickle
file=open('xgboost.pkl','wb')
pickle.dump(model,file)

In [ ]:
print("Values of the metrics of each model that was built:")    
for i,entry in enumerate(results):
    print(f"{i+1}:{entry}")